# UK Sanctions List – Transformation Notebook

This notebook converts the raw UK sanctions list into two outputs:

- **`sanctioned_parties_master.csv`**: one row per sanctioned party
- **`sanctioned_names.csv`**: one row per screening name / alias



## How this notebook is organized

1. Load the raw file carefully
2. Profile the dataset and identify data quality issues
3. Normalize names, dates, and repeated multi-value fields
4. Build an entity-level master dataset
5. Build a name-level screening dataset
6. Export files and summarize findings


## Data Retrieval

The source CSV has the **report-date on line 1**, and the real header starts on line 2.
That means we should load it with `skiprows=1` rather than treating the first line as the header.


In [1]:
import csv
import json
import re
from datetime import datetime
from pathlib import Path
import pandas as pd


INPUT_PATH = Path('UK-Sanctions-List.csv')
OUTPUT_ROOT = Path('notebook_output')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_folder_name = datetime.now().strftime('output_%b-%d_%H%M')
OUTPUT_DIR = OUTPUT_ROOT / run_folder_name
suffix = 1
while OUTPUT_DIR.exists():
    OUTPUT_DIR = OUTPUT_ROOT / f"{run_folder_name}_{suffix:02d}"
    suffix += 1
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)


report_date = pd.read_csv(INPUT_PATH, nrows=0).columns[0]
print( report_date)

Report Date: 29-Apr-2026


## Quick profiling

Initial exploratory checks to understand:
- Dataset structure and level of normalisation (denormalised sanctions records)
- Presence of duplicate rows
- Completeness of key identifiers (e.g. OFSI Group ID)
- Name field coverage and sparsity
- Distribution and quality of Date of Birth formats
- Early data quality risks before transformation

In [2]:
raw_preview = pd.read_csv(INPUT_PATH, skiprows=1, dtype=str)

print("Shape:", raw_preview.shape)
print("\nColumns:\n", raw_preview.columns.tolist())
print("\nSample:")
print(raw_preview.head(3))

# Basic missingness check
print("\nMissing values (top 10):")
print(raw_preview.isna().sum().sort_values(ascending=False).head(10))

Shape: (57033, 58)

Columns:
 ['Last Updated', 'Unique ID', 'OFSI Group ID', 'UN Reference Number', 'Name 6', 'Name 1', 'Name 2', 'Name 3', 'Name 4', 'Name 5', 'Name type', 'Alias strength', 'Title', 'Name non-latin script', 'Non-latin script type', 'Non-latin script language', 'Regime Name', 'Designation Type', 'Designation source', 'Sanctions Imposed', 'Other Information', 'UK Statement of Reasons', 'Address Line 1', 'Address Line 2', 'Address Line 3', 'Address Line 4', 'Address Line 5', 'Address Line 6', 'Address Postal Code', 'Address Country', 'Phone number', 'Website', 'Email address', 'Date Designated', 'D.O.B', 'Nationality(/ies)', 'National Identifier number', 'National Identifier additional information', 'Passport number', 'Passport additional information', 'Position', 'Gender', 'Town of birth', 'Country of birth', 'Type of entity', 'Subsidiaries', 'Parent company', 'Business registration number (s)', 'IMO number', 'Current owner/operator (s)', 'Previous owner/operator (s)', 

In [3]:
def normalize_space(value):
    """Standardise whitespace and null handling"""
    value = '' if value is None else str(value)
    value = value.replace('\u00a0', ' ')
    """non-breaking space"""
    value = re.sub(r'\s+', ' ', value)
    return value.strip()


def classify_dob(value):
    """
    Classify DOB formats to assess data quality:
    - Full (usable)
    - Masked (partial / redacted)
    - Multiple (multi-value field)
    - Year only (weak signal)
    - Blank (missing)
    """
    text = normalize_space(value)

    if not text:
        return 'Blank'
    if re.fullmatch(r'\d{2}/\d{2}/\d{4}', text):
        return 'Full'
    if re.search(r'dd|mm|yy|00', text, flags=re.IGNORECASE):
        return 'Masked'
    if '|' in text:
        return 'Multiple'
    if re.fullmatch(r'\d{4}', text):
        return 'Year only'
    return 'Other'

In [4]:
raw = pd.read_csv(INPUT_PATH, skiprows=1, dtype=str, keep_default_na=False)

# Reconstruct name before profiling. For organisations and ships, the usable name is often in Name 6,
# so checking Name 1 alone overstates blank-name issues
RAW_NAME_PART_COLUMNS = ['Title', 'Name 1', 'Name 2', 'Name 3', 'Name 4', 'Name 5', 'Name 6']
raw_profile_name = raw[RAW_NAME_PART_COLUMNS].apply(lambda row: ' '.join(normalize_space(x) for x in row if normalize_space(x)), axis=1)
raw_profile_non_latin = raw['Name non-latin script'].map(normalize_space)

profile = {
    'raw_rows': int(len(raw)),
    'exact_duplicate_rows': int(raw.duplicated().sum()),
    'missing_ofsi_group_id_rows': int((raw['OFSI Group ID'].str.strip() == '').sum()),
    'blank_name_1_rows': int((raw['Name 1'].str.strip() == '').sum()),
    'blank_reconstructed_name_rows': int(((raw_profile_name == '') & (raw_profile_non_latin == '')).sum()),
    'duplicate_ofsi_group_ids': int(raw['OFSI Group ID'].duplicated().sum())
}

dob_types = raw['D.O.B'].map(classify_dob)

dob_profile = {
    'full_dob_rows': int((dob_types == 'Full').sum()),
    'masked_dob_rows': int((dob_types == 'Masked').sum()),
    'multiple_dob_rows': int((dob_types == 'Multiple').sum()),
    'year_only_dob_rows': int((dob_types == 'Year only').sum()),
    'blank_dob_rows': int((dob_types == 'Blank').sum()),
}
print("\nStructural Profile")
print(profile)
print("\nDOB Quality Profile")
print(dob_profile)

name_1_coverage_ratio = (raw['Name 1'].str.strip() != '').mean()
reconstructed_name_coverage_ratio = ((raw_profile_name != '') | (raw_profile_non_latin != '')).mean()
print("\nProportion of rows with Name 1 populated:", name_1_coverage_ratio)
print("Proportion of rows with reconstructed Latin or non-Latin name:", reconstructed_name_coverage_ratio)

key_fields = [
    'OFSI Group ID',
    'Name 1',
    'Name 6',
    'D.O.B',
    'Nationality(/ies)',
    'Passport number'
]

print("\nKey field completeness")
for col in key_fields:
    completeness = (raw[col].str.strip() != '').mean()
    print(f"{col}: {completeness:.3f}")


Structural Profile
{'raw_rows': 57033, 'exact_duplicate_rows': 650, 'missing_ofsi_group_id_rows': 2264, 'blank_name_1_rows': 33142, 'blank_reconstructed_name_rows': 0, 'duplicate_ofsi_group_ids': 51895}

DOB Quality Profile
{'full_dob_rows': 16585, 'masked_dob_rows': 8038, 'multiple_dob_rows': 0, 'year_only_dob_rows': 186, 'blank_dob_rows': 32224}

Proportion of rows with Name 1 populated: 0.4188978310802518
Proportion of rows with reconstructed Latin or non-Latin name: 1.0

Key field completeness
OFSI Group ID: 0.960
Name 1: 0.419
Name 6: 0.998
D.O.B: 0.435
Nationality(/ies): 0.382
Passport number: 0.186


Key insights: 
1. Highly denormalised dataset
- Same entity repeated across multiple rows
- This is expected for sanctions data (1-to-many structure)

2. Duplicate rows exist
- Exact duplicate rows are present
- Suggests ingestion duplication OR repeated source rows
- Worth removing before aggregation

3. OFSI Group ID missing
- Fallback to Unique ID is required

4. Name fields are denormalised across Name 1 to Name 6
- `Name 1` alone is sparse and is not a reliable completeness check
- Organisations and ships often carry their usable name in `Name 6`
- Reconstructed Latin name plus non-Latin name is the right screening-name coverage metric

5. DOB quality is heavily fragmented
- DOB can't be relied on as a primary matching field
- Raw DOB text should be preserved alongside parsed ISO DOB and extracted years

## Data Transformation

This stage converts the raw denormalised sanctions dataset into structured entity-level representations.

Key operations include:
- Construction of a stable entity key (OFSI Group ID or fallback Unique ID)
- Aggregation of multi-column name and address fields into unified representations
- Standardisation and parsing of dates (DOB, designation dates, birth years)
- De-duplication of repeated values within pipe-delimited and multi-row fields
- Feature engineering to support downstream entity matching and screening.

### Helper Functions

In [5]:
RAW_DATE_RE = re.compile(r'^(\d{2})/(\d{2})/(\d{4})$')

# Detects duplicated sanction text in pipe-delimited fields
REPEATED_SANCTION_PATTERN = re.compile(r'(^|\|)\s*([^|]+?)\s*\|\s*\2\s*(\||$)', flags=re.IGNORECASE)

# Multi-column name and address reconstruction (denormalised structure)
NAME_PART_COLUMNS = ['Title', 'Name 1', 'Name 2', 'Name 3', 'Name 4', 'Name 5', 'Name 6']
ADDRESS_COLUMNS = ['Address Line 1', 'Address Line 2', 'Address Line 3', 'Address Line 4', 'Address Line 5', 'Address Line 6', 'Address Postal Code', 'Address Country']

# Useful organisation and vessel attributes for customer/supplier screening.
ENTITY_SHIP_COLUMNS = [
    'Type of entity',
    'Business registration number (s)',
    'Subsidiaries',
    'Parent company',
    'IMO number',
    'Type of ship',
    'Current owner/operator (s)',
    'Previous owner/operator (s)',
    'Current believed flag of ship',
    'Previous flags',
    'Tonnage of ship',
    'Length of ship',
    'Year Built',
    'Hull identification number (HIN)',
]


def normalize_name_type(value):
    """
    Standardise inconsistent source values such as:
    - Primary Name / Primary name
    - Primary Name Variation / Primary name variation
    - Alias / ALias
    """
    text = normalize_space(value).casefold()
    mapping = {
        'primary name': 'primary_name',
        'primary name variation': 'primary_name_variation',
        'alias': 'alias',
    }
    return mapping.get(text, text.replace(' ', '_'))


def parse_raw_date(value):
    """
    Convert DD/MM/YYYY → YYYY-MM-DD for standardisation.
    """
    text = normalize_space(value)
    m = RAW_DATE_RE.match(text)
    if not m:
        return ''
    d, mn, y = m.groups()
    return f'{y}-{mn}-{d}'


def split_pipe_values(values):
    """
    Normalise pipe-delimited sanction fields.
    """
    seen, out = set(), []
    for value in values:
        for part in str(value).split('|'):
            p = normalize_space(part)
            if not p:
                continue
            k = p.casefold()
            if k not in seen:
                seen.add(k)
                out.append(p)
    return out


def unique_join(values, casefold):
    """
    Generic deduplication and concatenation utility.
    """
    seen, out = set(), []
    for value in values:
        v = normalize_space(value)
        if not v:
            continue
        k = v.casefold() if casefold else v
        if k not in seen:
            seen.add(k)
            out.append(v)
    return ' | '.join(out)


def first_nonblank(values):
    """Return the first non-empty value in a list."""
    for value in values:
        v = normalize_space(value)
        if v:
            return v
    return ''


def extract_years(values):
    """
    Extract 4-digit years from messy DOB fields.
    """
    seen, out = set(), []
    for value in values:
        for year in re.findall(r'(?<!\d)(?:18|19|20)\d{2}(?!\d)', str(value)):
            if year not in seen:
                seen.add(year)
                out.append(year)
    return ' | '.join(out)

### Main Transformation

In [6]:
def load_raw(path):
    df = pd.read_csv(path, skiprows=1, dtype=str, keep_default_na=False)

    # Standardise whitespace across entire dataset
    for col in df.columns:
        df[col] = df[col].map(normalize_space)

    # OFSI Group ID is primary entity key and Unique ID used as a fallback when missing
    df['record_id'] = df['OFSI Group ID'].where(
        df['OFSI Group ID'].str.strip() != '',
        df['Unique ID']
    )

    # Normalise Name type once so later filtering is case-insensitive and consistent
    df['name_type_normalized'] = df['Name type'].map(normalize_name_type)

    # Name and address reconstruction. Name can be spread across Title/Name 1-6;
    # organisations and ships commonly use Name 6 rather than Name 1.
    df['full_name'] = df[NAME_PART_COLUMNS].apply(
        lambda row: ' '.join(normalize_space(x) for x in row if normalize_space(x)),
        axis=1,
    )
    df['full_address'] = df[ADDRESS_COLUMNS].apply(
        lambda row: ', '.join(normalize_space(x) for x in row if normalize_space(x)),
        axis=1,
    )

    # DOB enrichments: preserve raw value in outputs and add parsed/date-derived fields
    df['dob_type'] = df['D.O.B'].map(classify_dob)
    df['date_of_birth'] = df['D.O.B'].apply(parse_raw_date)
    df['year_of_birth'] = df['D.O.B'].apply(lambda x: extract_years([x]))

    # Entity quality features
    df['entity_size'] = df['record_id'].map(df['record_id'].value_counts())
    df['missing_name_flag'] = (df['full_name'].str.strip() == '') & (df['Name non-latin script'].str.strip() == '')
    df['missing_address_flag'] = df['full_address'].str.strip() == ''
    df['sanctions_repeated_flag'] = df['Sanctions Imposed'].apply(
        lambda x: bool(REPEATED_SANCTION_PATTERN.search(str(x)))
    )
    
    return df

In [7]:
raw = load_raw(INPUT_PATH)

# Entity size distribution
print("\nEntity size stats:")
print(raw['entity_size'].describe())

entity_sizes = (
    raw.groupby("record_id")
       .size()
       .sort_values(ascending=False)
)

print("Largest entities (alias-heavy / complex):")
display(entity_sizes.head(10))
print('\n Quantile Distribution')
print(entity_sizes.quantile([0.5, 0.75, 0.9, 0.95, 0.99]))
# Missing data signals
print("\nMissing name flag rate:")
print(raw['missing_name_flag'].mean())

print("\nMissing address flag rate:")
print(raw['missing_address_flag'].mean())

# Sanctions text quality issues
print("\nRows with repeated sanction text issues:")
print(raw['sanctions_repeated_flag'].mean())

# Entity-level structure recap
print("\nEntity summary:")
print("Total rows:", len(raw))
print("Unique entities:", raw['record_id'].nunique())
print("Avg rows per entity:", len(raw) / raw['record_id'].nunique())


Entity size stats:
count    57033.000000
mean       942.897884
std       1268.922992
min          1.000000
25%         15.000000
50%        152.000000
75%       1292.000000
max       3780.000000
Name: entity_size, dtype: float64
Largest entities (alias-heavy / complex):


record_id
10659    3780
11559    3528
12268    2800
12816    1920
12999    1440
16718    1292
7860     1260
13344    1200
12429    1144
11199    1080
dtype: int64


 Quantile Distribution
0.50     2.0
0.75     4.0
0.90     9.0
0.95    20.0
0.99    96.0
dtype: float64

Missing name flag rate:
0.0

Missing address flag rate:
0.17973804639419283

Rows with repeated sanction text issues:
0.023933512177160592

Entity summary:
Total rows: 57033
Unique entities: 6035
Avg rows per entity: 9.450372825186413


Key Insights
1. Highly denormalised dataset
- 57,033 rows map to only 6,035 unique entities
- Average ~9.5 rows per entity confirms strong 1-to-many structure
  
2. Extreme entity skew
- Entity sizes range from 1 to 3,780 rows
- A small number of entities dominate the dataset (alias-heavy / highly duplicated records)
- The dataset has a heavily skewed entity distribution, where most sanctioned entities have only a few associated rows, but a small number of entities have thousands of records due to extensive aliasing and address duplication.

3. Low missingness in core identifiers
- Name fields are almost fully populated (~0.1% missing)
- Names are reliable as primary matching signals

4. Weaker address coverage
- ~18% of records are missing address data
- Addresses should be treated as secondary or supporting features

5. Minor data quality issues
- ~2.4% of rows show repeated sanction text patterns, suggesting mild ingestion inconsistencies

## Building Outputs

The key design decision is to build:

- **one master row per sanctioned party**, keyed by `OFSI Group ID` where available and `Unique ID` otherwise
- **one screening-name row per usable name variant**, so aliases and non-Latin names remain available for matching


In [8]:
def build_outputs(raw):

    # Remove exact duplicate ingestion rows
    dedup = raw.drop_duplicates().copy()
    
    groups = {}
    name_rows = []

    grouped = dedup.groupby('record_id', sort=True)

    for record_id, g in grouped:
        # Convert group to dict of lists for flexible aggregation
        vals = {c: g[c].tolist() for c in g.columns}

        # Name Handling: use normalised name type, avoiding case-sensitive misses in the source.
        primary_mask = g['name_type_normalized'].isin(['primary_name', 'primary_name_variation'])
        alias_mask = g['name_type_normalized'].isin(['alias', 'primary_name_variation'])

        primary_names = [x for x in g.loc[primary_mask, 'full_name'].tolist() if normalize_space(x)]

        # All name variants (Latin + non-Latin)
        all_names_list = [
            x for x in g['full_name'].tolist() + g['Name non-latin script'].tolist()
            if normalize_space(x)
        ]

        # Alias-only subset for fuzzy matching prioritisation
        alias_names_list = [
            x for x in g.loc[alias_mask, 'full_name'].tolist()
            + g.loc[alias_mask, 'Name non-latin script'].tolist()
            if normalize_space(x)
        ]

        all_names_joined = unique_join(all_names_list, casefold=True)

        # Countries
        associated = []
        for col in ['Address Country', 'Country of birth', 'Nationality(/ies)', 'Current believed flag of ship', 'Previous flags']:
            associated.extend([x for x in vals[col] if normalize_space(x)])

        # Data quality flags
        flags = []

        # Missing identifiers
        if not first_nonblank(vals['OFSI Group ID']):
            flags.append('missing_ofsi_group_id')

        # DOB quality
        dob_types = [classify_dob(x) for x in vals['D.O.B']]
        if 'Blank' in dob_types:
            flags.append('missing_dob')
        if 'Masked' in dob_types:
            flags.append('masked_dob')
        if 'Multiple' in dob_types:
            flags.append('multiple_dob')

        # Missing name
        if not all_names_list:
            flags.append('missing_name')

        # Missing country
        if not associated:
            flags.append('missing_country')

        # Missing identifiers (passport / national ID / registration / IMO)
        identifier_values = (
            vals['Passport number']
            + vals['National Identifier number']
            + vals['Business registration number (s)']
            + vals['IMO number']
        )
        if not any(normalize_space(x) for x in identifier_values):
            flags.append('no_identifiers')

        # Repeated sanctions text
        if any(REPEATED_SANCTION_PATTERN.search(str(x) or '') for x in vals['Sanctions Imposed']):
            flags.append('possible_repeated_sanction_text')

        # Master entity record (one row per sanctioned entity)
        ofsi_id = first_nonblank(vals['OFSI Group ID'])
        unique_id = first_nonblank(vals['Unique ID'])

        # Explicit key source tracking
        key_source = 'OFSI Group ID' if ofsi_id else 'Unique ID'

        # Simple risk scoring hook
        risk_score = len(flags)
        
        master_row = {
            'record_id': record_id,
            
            'ofsi_group_id': ofsi_id,
            'unique_id': unique_id,
            'key_source': key_source,
            
            'primary_name': primary_names[0] if primary_names else (all_names_list[0] if all_names_list else ''),
            'all_names': all_names_joined,
            'all_aliases': unique_join(alias_names_list, casefold=True),
            
            'date_of_births_raw': unique_join(vals['D.O.B'], casefold=True),
            'date_of_births': unique_join([parse_raw_date(x) for x in vals['D.O.B']], casefold=True),
            'dob_years': extract_years(vals['D.O.B']),
            
            'associated_countries': unique_join(associated, casefold=True),
            'addresses': unique_join(vals['full_address'], casefold=True),
            
            'passport_numbers': unique_join(vals['Passport number'], casefold=True),
            'national_identifier_numbers': unique_join(vals['National Identifier number'], casefold=True),
            
            'sanctions_imposed': ' | '.join(split_pipe_values(vals['Sanctions Imposed'])),
            'designation_type': first_nonblank(vals['Designation Type']),
            
            'source_row_count': len(g),
            
            'data_quality_flags': ' | '.join(flags),
            'risk_score': risk_score
        }

        # Add organisation and vessel attributes for entity/ship matching.
        for col in ENTITY_SHIP_COLUMNS:
            master_row[col.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')] = unique_join(vals[col], casefold=True)

        groups[record_id] = master_row

        # Screening table (name-level matching dataset)
        ng = g[(g['full_name'].str.strip() != '') | (g['Name non-latin script'].str.strip() != '')].copy()

        if not ng.empty:
            ng['name_for_screening'] = ng['full_name'].where(
                ng['full_name'].str.strip() != '',
                ng['Name non-latin script']
            )

            ng['date_of_birth_iso'] = ng['D.O.B'].map(parse_raw_date)
            ng['date_of_birth_raw'] = ng['D.O.B']
            ng['dob_year'] = ng['D.O.B'].apply(lambda x: extract_years([x]))
            
            # Prevent duplicate name rows per entity while preserving name type and alias strength.
            ng = ng.drop_duplicates(
                subset=['record_id', 'name_type_normalized', 'Alias strength', 'name_for_screening', 'D.O.B']
            )
            name_rows.append(ng)

    master = pd.DataFrame(list(groups.values()))
    names = pd.concat(name_rows, ignore_index=True) if name_rows else pd.DataFrame()

    return dedup, master, names


dedup, master, names = build_outputs(raw)

## Data Quality Assessment

This section evaluates data quality at both **row level** and **entity level**. The distinction matters because the source is highly denormalised: quality can look weak at row level while still being acceptable once records are consolidated to sanctioned-party level.

The assessment focuses on five areas:
- structural duplication and entity skew
- completeness of names, addresses, dates of birth, and identifiers
- consistency of source-controlled categorical fields
- screening usability after name reconstruction and alias explosion
- implications for downstream sanctions matching

In [9]:
summary = {
    'report_date': report_date,
    'raw_rows': int(len(raw)),
    'rows_after_dedup': int(len(dedup)),
    'master_records': int(len(master)),
    'screening_name_rows': int(len(names)),
    'designation_type_counts': master['designation_type'].value_counts().to_dict(),
}

print(summary)

{'report_date': 'Report Date: 29-Apr-2026', 'raw_rows': 57033, 'rows_after_dedup': 56383, 'master_records': 6035, 'screening_name_rows': 17119, 'designation_type_counts': {'Individual': 3872, 'Entity': 1531, 'Ship': 632}}


In [10]:
# Row-level quality
print("Row-level quality")
print("- Raw rows:", f"{len(raw):,}")
print("- Exact duplicate rows:", f"{raw.duplicated().sum():,}")
print("- Rows missing OFSI Group ID:", f"{(raw['OFSI Group ID'].eq('')).sum():,}")
print("- Rows with blank Name 1:", f"{(raw['Name 1'].eq('')).sum():,}")
print("- Rows with blank reconstructed name:", f"{((raw['full_name'].eq('')) & (raw['Name non-latin script'].eq(''))).sum():,}")
print("- Rows with blank full address:", f"{(raw['full_address'].eq('')).sum():,}")
print("- Rows with blank DOB:", f"{(raw['D.O.B'].eq('')).sum():,}")
print("- Rows with fully parsed DOB:", f"{raw['D.O.B'].map(classify_dob).eq('Full').sum():,}")
print("- Rows with masked DOB:", f"{raw['D.O.B'].map(classify_dob).eq('Masked').sum():,}")

# Entity-level quality
print("\nEntity-level quality")
print("- Master records:", f"{len(master):,}")
print("- Median source rows per entity:", master['source_row_count'].median())
print("- 95th percentile source rows per entity:", master['source_row_count'].quantile(0.95))
print("- Maximum source rows for a single entity:", int(master['source_row_count'].max()))
print("- Entities missing any address %:", round((master['addresses'] == '').mean() * 100, 2))
print("- Entities missing any DOB %:", round((master['date_of_births_raw'] == '').mean() * 100, 2))
print("- Entities missing any fully parsed DOB %:", round((master['date_of_births'] == '').mean() * 100, 2))
print(
    "- Entities missing all identifier types %:",
    round(
        (
            (master['passport_numbers'] == '')
            & (master['national_identifier_numbers'] == '')
            & (master['business_registration_number_s'] == '')
            & (master['imo_number'] == '')
        ).mean() * 100,
        2,
    ),
)

# Screening usability
print("\nScreening usability")
name_counts = names.groupby('record_id').size()
print("- Screening-name rows:", f"{len(names):,}")
print("- Median names per entity:", name_counts.median())
print("- 95th percentile names per entity:", name_counts.quantile(0.95))
print("- Maximum names for a single entity:", int(name_counts.max()))

# Categorical consistency and population mix
print("\nCategorical consistency and population mix")
print("- Normalised designation types:")
print(master['designation_type'].value_counts())
print("\n- Raw Name type values before normalisation:")
print(raw['Name type'].replace('', pd.NA).value_counts(dropna=True).head(10))
print("\n- Source entity type values:")
print(master['type_of_entity'].replace('', pd.NA).value_counts(dropna=True).head(10))

# Risk score distribution
print("\nRisk score distribution")
print(master['risk_score'].describe())

Row-level quality
- Raw rows: 57,033
- Exact duplicate rows: 650
- Rows missing OFSI Group ID: 2,264
- Rows with blank Name 1: 33,142
- Rows with blank reconstructed name: 0
- Rows with blank full address: 10,251
- Rows with blank DOB: 32,224
- Rows with fully parsed DOB: 16,585
- Rows with masked DOB: 8,038

Entity-level quality
- Master records: 6,035
- Median source rows per entity: 2.0
- 95th percentile source rows per entity: 20.0
- Maximum source rows for a single entity: 3780
- Entities missing any address %: 47.97
- Entities missing any DOB %: 45.93
- Entities missing any fully parsed DOB %: 55.14
- Entities missing all identifier types %: 68.17

Screening usability
- Screening-name rows: 17,119
- Median names per entity: 1.0
- 95th percentile names per entity: 9.0
- Maximum names for a single entity: 144

Categorical consistency and population mix
- Normalised designation types:
designation_type
Individual    3872
Entity        1531
Ship           632
Name: count, dtype: int64

Key Insights
1. The source is extremely denormalised and highly skewed
- `57,033` raw rows collapse to `6,035` unique entities, confirming a strong many-to-one structure
- The median entity has only `2` source rows, but the 95th percentile is `20` and the largest entity has `3,780` rows
- This means aggregation is not optional: it is a prerequisite for any usable downstream screening dataset

2. Duplicate ingestion noise exists, but it is not the main quality problem
- There are `650` exact duplicate rows in the raw file
- These should be removed before aggregation, but the more important issue is structural repetition across aliases, addresses, and entity fragments

3. Name coverage is strong after reconstruction, but raw column-level coverage is misleading
- `Name 1` is blank on `33,142` rows, which could wrongly suggest severe name incompleteness
- After reconstructing names across Title and Name 1-6 and retaining non-Latin script names, there are `0` rows and `0` entities with no usable name
- This confirms names are the strongest matching signal in the dataset once the denormalised structure is handled correctly

4. Identifier quality is the main screening weakness
- `2,264` rows are missing `OFSI Group ID`, so fallback to `Unique ID` is necessary for a stable entity key
- At entity level, `68.17%` of sanctioned parties have no passport number, national identifier, business registration number, or IMO number
- Only `31.83%` of entities retain at least one of those stronger identifiers, which materially limits deterministic matching

5. DOB coverage is mixed and full DOB coverage is materially weaker than raw availability suggests
- `32,224` rows have no DOB at all and only `16,585` rows contain a fully parseable DOB
- At entity level, `45.93%` of entities have no DOB and `55.15%` have no fully parsed DOB
- DOB is therefore useful as supporting context, but not reliable enough to act as a primary screening key

6. Address coverage is meaningful but incomplete
- `10,251` raw rows have no reconstructed address
- At entity level, `47.97%` of entities have no usable aggregated address
- Address should therefore be treated as contextual enrichment rather than a consistently available core attribute

7. Name-type inconsistencies would create data loss if left untreated
- The raw file mixes values such as `Primary Name`, `Primary name`, `Primary Name Variation`, `Primary name variation`, `Alias`, and `ALias`
- Normalising these values is necessary to avoid missing primary names and aliases in both outputs

8. The name-level screening output is justified by the alias structure
- The final screening dataset contains `17,119` unique name rows
- The median entity has `1` screening name, the 95th percentile has `9`, and the maximum is `144`
- This confirms that screening on only one master-name field would lose meaningful alias coverage for a non-trivial subset of entities


Conclusion:

The transformed dataset is structurally sound at entity level, but identity resolution remains constrained by weak identifier coverage and fragmented DOB data. In practice, that means sanctions screening should rely primarily on name-based matching, supported by contextual enrichment such as country, address, nationality, vessel attributes, and partial date-of-birth evidence rather than deterministic identifiers alone.

## Export outputs



In [11]:
master_path = OUTPUT_DIR / 'sanctioned_parties_master.csv'
names_path = OUTPUT_DIR / 'sanctioned_names.csv'
summary_path = OUTPUT_DIR / 'run_summary.json'

master.to_csv(master_path, index=False, quoting=csv.QUOTE_MINIMAL)
names.to_csv(names_path, index=False, quoting=csv.QUOTE_MINIMAL)
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(master_path)
print(names_path)
print(summary_path)

notebook_output/output_May-04_1601/sanctioned_parties_master.csv
notebook_output/output_May-04_1601/sanctioned_names.csv
notebook_output/output_May-04_1601/run_summary.json


## Notes

- The source is **denormalized**, so one person or entity appears on many rows.
- The transformation uses **`OFSI Group ID` as the preferred entity key** and falls back to `Unique ID` when needed.
- `Name type` is normalised before filtering, so inconsistent values such as `Primary Name`, `Primary name`, `Primary Name Variation`, and `ALias` do not cause missed names.
- `Name 1` alone is not a valid completeness metric because organisation and vessel names often sit in `Name 6`; the notebook now assesses reconstructed Latin names plus non-Latin names.
- Partial DOBs such as `dd/mm/1958` are still useful, so the pipeline keeps raw DOB text, parsed full DOBs where possible, and extracted DOB years.
- Name screening should happen on the **exploded alias dataset**, not only on the master file.
- Useful screening fields were preserved beyond names and DOBs, including nationalities, associated countries, personal identifiers, business registration numbers, parent/subsidiary fields, IMO numbers, ship type, owner/operator, flags, tonnage, length, year built, and HIN.

## Record ID Comparison

In [13]:
key_comparison = raw.copy()

key_comparison["record_id_ofsi_first"] = key_comparison["OFSI Group ID"].where(
    key_comparison["OFSI Group ID"].str.strip() != "",
    key_comparison["Unique ID"]
)

key_comparison["record_id_unique_only"] = key_comparison["Unique ID"]

ofsi_first_sizes = key_comparison.groupby("record_id_ofsi_first").size()
unique_only_sizes = key_comparison.groupby("record_id_unique_only").size()

comparison = pd.DataFrame(
    {
        "metric": [
            "Entities",
            "Median entity size",
            "Max entity size",
        ],
        "OFSI-first": [
            ofsi_first_sizes.shape[0],
            ofsi_first_sizes.median(),
            ofsi_first_sizes.max(),
        ],
        "Unique-only": [
            unique_only_sizes.shape[0],
            unique_only_sizes.median(),
            unique_only_sizes.max(),
        ],
    }
)

display(comparison)

,metric,OFSI-first,Unique-only
0,Entities,6035.0,6046.0
1,Median entity size,2.0,2.0
2,Max entity size,3780.0,3780.0


In [16]:
df1 = raw.copy()

df1 = df1[df1["OFSI Group ID"].str.strip() != ""]

multi_unique = (
    df1.groupby("OFSI Group ID")["Unique ID"]
    .nunique()
    .reset_index()
)

multi_unique = multi_unique[multi_unique["Unique ID"] > 1]

print("Number of entities with multiple Unique IDs:", len(multi_unique))

print(multi_unique)

Number of entities with multiple Unique IDs: 11
     OFSI Group ID  Unique ID
1355         13748          2
1488         13897          2
1509         13918          2
1510         13919          2
1511         13920          2
1512         13921          2
1513         13922          2
1514         13923          2
1515         13924          2
1516         13925          2
4796          6995          2
